# Session 14 · Green status, wrong data

Crown Street Markets, a fictional 40-store Charlotte grocery chain, loads last night's register baskets into BigQuery every night. The replenishment dashboard reads them at 06:00.

In [ ]:
source ~/dsba6190-live-demo-14/env.sh && echo "$DATASET" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The nightly load succeeds

In [ ]:
bq --project_id="$PROJECT" --format=json show "$PROJECT:$DATASET.store_sales" \
  | jq '{labels, timePartitioning, numRows}'

In [ ]:
nightly_load "gs://$BUCKET/raw/dt=$LAST_NIGHT/sales.csv"

In [ ]:
q "SELECT sale_date, COUNT(*) AS \`rows\`, ROUND(SUM(basket_value)) AS sales_usd FROM \`$DS.store_sales\`
   WHERE sale_date >= DATE_SUB('$LAST_NIGHT', INTERVAL 3 DAY) GROUP BY 1 ORDER BY 1"

## Step 2 · Four checks in SQL

In [ ]:
cat sql/checks.sql

In [ ]:
checks

## Step 3 · Break it three ways, and every load succeeds

In [ ]:
nightly_load "gs://$BUCKET/incoming/one-store.csv"

In [ ]:
checks

In [ ]:
nightly_load "gs://$BUCKET/incoming/renamed-column.csv"

In [ ]:
checks

In [ ]:
nightly_load "gs://$BUCKET/raw/dt=$LAST_NIGHT/sales.csv"
PIPELINE=fix q "ALTER TABLE \`$DS.store_sales\` DROP COLUMN basket_total"

In [ ]:
nightly_load "gs://$BUCKET/incoming/north-nulls.csv"

In [ ]:
checks

In [ ]:
q "SELECT region, COUNT(*) AS \`rows\`, COUNTIF(basket_value IS NULL) AS null_values,
          ROUND(AVG(basket_value), 2) AS mean_basket
   FROM \`$DS.store_sales\` WHERE sale_date = '$LAST_NIGHT' GROUP BY 1 ORDER BY 1"

In [ ]:
nightly_load "gs://$BUCKET/raw/dt=$LAST_NIGHT/sales.csv"
checks

## Step 4 · Make one check a symptom alert

In [ ]:
log_checks

In [ ]:
cat metric.yaml
gcloud logging metrics create "$METRIC" --config-from-file=metric.yaml --project "$PROJECT"

In [ ]:
CHANNEL=$(gcloud beta monitoring channels create --project "$PROJECT" \
    --display-name="Replenishment on-call (run $SUFFIX)" --type=email \
    --channel-labels=email_address=instructor@example.edu --format='value(name)')
echo "$CHANNEL" | tee channel.name

In [ ]:
until POLICY=$(gcloud monitoring policies create --project "$PROJECT" --policy-from-file=policy.json \
        --notification-channels="$CHANNEL" --format='value(name)' 2>/dev/null); do
  echo 'The new metric is not visible to Monitoring yet. Retrying in 30 s.'; sleep 30
done
echo "$POLICY" | tee policy.name

In [ ]:
gcloud monitoring policies describe "$POLICY" --project "$PROJECT" \
  --format="yaml(displayName,enabled,conditions[0].conditionThreshold.filter,conditions[0].conditionThreshold.comparison,conditions[0].conditionThreshold.thresholdValue,notificationChannels.len())"

In [ ]:
nightly_load "gs://$BUCKET/incoming/one-store.csv" >/dev/null && log_checks

In [ ]:
sleep 10
gcloud logging read "logName=\"projects/$PROJECT/logs/$LOG\" AND jsonPayload.status=\"RED\"" \
  --project "$PROJECT" --limit=1 --format=json | jq '.[0] | {severity, logName, jsonPayload}'

In [ ]:
nightly_load "gs://$BUCKET/raw/dt=$LAST_NIGHT/sales.csv" >/dev/null && checks

## Step 5 · The SLO and the error budget

In [ ]:
q "SELECT sale_date, state, finished_at, rows_loaded, corrected_at FROM \`$DS.load_runs\`
   WHERE TIME(finished_at) > '06:00:00' OR corrected_at IS NOT NULL ORDER BY 1"

In [ ]:
PIPELINE=slo_report q "$(sed "s/@@DS@@/$DS/g" sql/error-budget.sql)"

In [ ]:
red_count

## Step 6 · Delete by mistake, restore with time travel

In [ ]:
BEFORE=$(date -u +%FT%TZ); echo "recovery point $BEFORE"; sleep 2
PIPELINE=mistake q "DELETE FROM \`$DS.store_sales\`
   WHERE sale_date = '$LAST_NIGHT'
   -- AND register_id = 'CLT-040-R99'   the line that was meant to be here"

In [ ]:
checks

In [ ]:
PIPELINE=restore q "CREATE TEMP TABLE lost AS
   SELECT * FROM \`$DS.store_sales\` FOR SYSTEM_TIME AS OF TIMESTAMP '${BEFORE:?}' WHERE sale_date = '$LAST_NIGHT';
   INSERT INTO \`$DS.store_sales\` SELECT * FROM lost;"

In [ ]:
checks

In [ ]:
q "WITH ours AS (
     SELECT job_id, l.value AS step, start_time, end_time
     FROM \`region-us\`.INFORMATION_SCHEMA.JOBS AS j, UNNEST(j.labels) AS l
     WHERE l.key = 'pipeline' AND l.value IN ('mistake', 'restore') AND j.parent_job_id IS NULL AND EXISTS (SELECT 1 FROM UNNEST(j.labels) AS r WHERE r.key = 'run' AND r.value = '$SUFFIX')
       AND j.creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 2 HOUR))
   SELECT o.step, FORMAT_TIMESTAMP('%T', o.end_time) AS finished_utc,
          SUM(IF(j.statement_type = 'DELETE', j.dml_statistics.deleted_row_count, 0)) AS deleted,
          SUM(IF(j.statement_type = 'INSERT', j.dml_statistics.inserted_row_count, 0)) AS inserted,
          TIMESTAMP_DIFF(o.end_time, MIN(o.end_time) OVER (), SECOND) AS s_after_mistake
   FROM ours AS o
   JOIN \`region-us\`.INFORMATION_SCHEMA.JOBS AS j ON (j.job_id = o.job_id OR j.parent_job_id = o.job_id)
    AND j.creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 2 HOUR)
   GROUP BY o.step, o.start_time, o.end_time ORDER BY o.end_time"

In [ ]:
bq --project_id="$PROJECT" --location=US mk --dataset --label team:replenishment "$PROJECT:$BACKUP"
PIPELINE=backup q "CREATE SNAPSHOT TABLE \`$PROJECT.$BACKUP.store_sales_$LAST_ID\`
   CLONE \`$DS.store_sales\`
   OPTIONS (expiration_timestamp = TIMESTAMP_ADD(CURRENT_TIMESTAMP(), INTERVAL 30 DAY))"
bq --project_id="$PROJECT" ls "$PROJECT:$BACKUP"

## Step 7 · FinOps: labels, the cheapest fix, and a storage move nobody notices

In [ ]:
PIPELINE=replenishment_dashboard q "SELECT COUNT(*) AS store_days, ROUND(SUM(sales)) AS sales_usd FROM (
     SELECT sale_date, store_id, region, COUNT(DISTINCT basket_id) AS baskets,
            SUM(items) AS items, SUM(basket_value) AS sales
     FROM \`$DS.store_sales\`
     WHERE DATE_DIFF(CURRENT_DATE('America/New_York'), sale_date, DAY) BETWEEN 1 AND 7
     GROUP BY 1, 2, 3)"

In [ ]:
PIPELINE=replenishment_dashboard q "SELECT COUNT(*) AS store_days, ROUND(SUM(sales)) AS sales_usd FROM (
     SELECT sale_date, store_id, region, COUNT(DISTINCT basket_id) AS baskets,
            SUM(items) AS items, SUM(basket_value) AS sales
     FROM \`$DS.store_sales\`
     WHERE sale_date BETWEEN DATE_SUB(CURRENT_DATE('America/New_York'), INTERVAL 7 DAY)
                         AND DATE_SUB(CURRENT_DATE('America/New_York'), INTERVAL 1 DAY)
     GROUP BY 1, 2, 3)"

In [ ]:
q "SELECT l.value AS pipeline, COUNT(*) AS jobs,
          ROUND(SUM(total_bytes_processed) / POW(2, 20), 1) AS processed_mb,
          ROUND(SUM(IFNULL(total_bytes_billed, 0)) / POW(2, 20), 1) AS billed_mb
   FROM \`region-us\`.INFORMATION_SCHEMA.JOBS AS j, UNNEST(j.labels) AS l
   WHERE l.key = 'pipeline' AND j.parent_job_id IS NULL AND EXISTS (SELECT 1 FROM UNNEST(j.labels) AS r WHERE r.key = 'run' AND r.value = '$SUFFIX')
     AND j.creation_time > TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 DAY)
   GROUP BY 1 ORDER BY billed_mb DESC, jobs DESC"

In [ ]:
gcloud storage buckets update "gs://$BUCKET" --project "$PROJECT" --lifecycle-file=lifecycle.json
gcloud storage buckets describe "gs://$BUCKET" --project "$PROJECT" \
  --format='yaml(default_storage_class,labels,lifecycle_config)'

In [ ]:
OLD="gs://$BUCKET/raw/dt=$FIRST_NIGHT/sales.csv"
gcloud storage objects describe "$OLD" --project "$PROJECT" --format=json \
  | jq -r '"gs://\(.bucket)/\(.name)  \(.storage_class)  generation \(.generation)"'
gcloud storage objects update "$OLD" --project "$PROJECT" --storage-class=NEARLINE
gcloud storage objects describe "$OLD" --project "$PROJECT" --format=json \
  | jq -r '"gs://\(.bucket)/\(.name)  \(.storage_class)  generation \(.generation)"'

In [ ]:
gcloud storage cat "$OLD" --project "$PROJECT" | sed -n 1,3p

## Step 8 · Teardown, and verify nothing is left

In [ ]:
POLICY=${POLICY:-$(cat policy.name)}; CHANNEL=${CHANNEL:-$(cat channel.name)}
gcloud monitoring policies delete "${POLICY:?}" --project "$PROJECT" --quiet
gcloud beta monitoring channels delete "${CHANNEL:?}" --project "$PROJECT" --quiet

In [ ]:
gcloud logging metrics delete "${METRIC:?}" --project "$PROJECT" --quiet
gcloud logging logs delete "${LOG:?}" --project "$PROJECT" --quiet

In [ ]:
gcloud storage rm -r "gs://${BUCKET:?}" --project "$PROJECT" 2>&1 | tail -2

In [ ]:
bq --project_id="$PROJECT" rm -r -f -d "${DATASET:?}"
bq --project_id="$PROJECT" rm -r -f -d "${BACKUP:?}"

In [ ]:
printf '%-12s %s\n' datasets  $(bq --project_id="$PROJECT" ls --max_results=1000 | grep -c "$SUFFIX")
printf '%-12s %s\n' snapshots $(bq --project_id="$PROJECT" ls "$PROJECT:$BACKUP" 2>/dev/null | grep -c SNAPSHOT)
printf '%-12s %s\n' buckets   $(gcloud storage ls --project "$PROJECT" | grep -c "$SUFFIX")
printf '%-12s %s\n' metrics   $(gcloud logging metrics list --project "$PROJECT" --format='value(name)' | grep -c "$SUFFIX")
printf '%-12s %s\n' logs      $(gcloud logging logs list --project "$PROJECT" | grep -c "$SUFFIX")
printf '%-12s %s\n' policies  $(gcloud monitoring policies list --project "$PROJECT" --format='value(displayName)' | grep -c "$SUFFIX")
printf '%-12s %s\n' channels  $(gcloud beta monitoring channels list --project "$PROJECT" --format='value(displayName)' | grep -c "$SUFFIX")